In [1]:
!pip install -q shapely folium requests numpy


# BlastSafe DSS: Simulador de Dispersión Atmosférica
## 1. Instalación de Dependencias



## 2. Módulo Meteorológico (clima.py)

In [28]:
%%writefile clima.py
import requests

def obtener_pronostico(lat: float, lon: float, horas: int = 24) -> dict:
    headers = {"User-Agent": "(BlastSafePrototype, contacto@ejemplo.com)"}

    # Construimos la URL por partes para evitar recortes del navegador
    host_noaa = "https:" + "//api.weather.gov"
    ruta_puntos = f"{host_noaa}/points/{lat},{lon}"

    try:
        r_punto = requests.get(ruta_puntos, headers=headers, timeout=10)
        if r_punto.status_code == 200:
            forecast_url = r_punto.json()["properties"]["forecastHourly"]
            r_fore = requests.get(forecast_url, headers=headers, timeout=10)
            periodos = r_fore.json()["properties"]["periods"][:horas]

            dir_map = {
                "N": 0, "NNE": 22.5, "NE": 45, "ENE": 67.5,
                "E": 90, "ESE": 112.5, "SE": 135, "SSE": 157.5,
                "S": 180, "SSW": 202.5, "SW": 225, "WSW": 247.5,
                "W": 270, "WNW": 292.5, "NW": 315, "NNW": 337.5
            }

            tiempos, temps, vientos, dirs = [], [], [], []
            for p in periodos:
                tiempos.append(p["startTime"])
                t_c = (p["temperature"] - 32) * 5.0 / 9.0
                temps.append(round(t_c, 1))
                mph = float(p["windSpeed"].split().pop(0))
                vientos.append(round(mph * 0.44704, 2))
                dirs.append(dir_map.get(p["windDirection"], 315.0))

            print("Conectado en vivo a NOAA (National Weather Service).")
            return {
                "time": tiempos,
                "temperature_2m": temps,
                "temperature_120m": [round(t - 0.8, 1) for t in temps],
                "wind_speed_10m": vientos,
                "wind_speed_120m": [round(v * 1.25, 2) for v in vientos],
                "wind_direction_120m": dirs
            }
    except Exception as e:
        print(f"Fallo conexión NOAA ({e}). Usando datos locales de respaldo.")

    return {
        "time": list(["2026-11-25T08:00"]),
        "temperature_2m": list([8.5]),
        "temperature_120m": list([7.2]),
        "wind_speed_10m": list([4.2]),
        "wind_speed_120m": list([6.4]),
        "wind_direction_120m": list([315.0])
    }


Overwriting clima.py


## 3. Motor de Dispersión Gaussiana (dispersion.py)


In [29]:
%%writefile dispersion.py
import math
import numpy as np
from shapely.geometry import Polygon
from shapely.affinity import rotate

def calcular_pluma(lat: float, lon: float, masa_kg: float, ws: float, wd: float):
    u = max(ws, 1.5)
    plomo_mg = (masa_kg * 0.71 * 0.35) * 1e6
    H = 18.5 * ((masa_kg * 0.35) ** 0.25)
    vs = 0.12
    umbral = 0.43

    x_vals = np.linspace(50, 12000, 200)
    y_vals = np.linspace(-2500, 2500, 100)
    X, Y = np.meshgrid(x_vals, y_vals)

    sy = 0.08 * X * ((1 + 0.0001 * X) ** -0.5)
    sz = 0.06 * X * ((1 + 0.0002 * X) ** -0.5)
    H_eff = np.maximum(0.0, H - (vs * X / u))

    dep = ((plomo_mg * vs) / (2 * np.pi * u * sy * sz)) * \
          np.exp(-(Y**2) / (2 * sy**2)) * np.exp(-(H_eff**2) / (2 * sz**2))

    mask = dep >= umbral
    if not np.any(mask):
        return None

    sup, inf = [], []
    for i in range(len(x_vals)):
        col = mask[:, i]
        if np.any(col):
            indices = np.flatnonzero(col)
            sup.append((x_vals[i], y_vals[indices.max()]))
            inf.append((x_vals[i], y_vals[indices.min()]))

    if len(sup) < 3:
        return None

    poly = Polygon(sup + inf[::-1]).buffer(40)
    angulo = 90.0 - ((wd + 180.0) % 360.0)
    poly_rot = rotate(poly, angulo, origin=(0, 0))

    m_lat = 111320.0
    m_lon = 111320.0 * math.cos(math.radians(lat))
    coords = [(lon + x / m_lon, lat + y / m_lat) for x, y in poly_rot.exterior.coords]
    return Polygon(coords)


Overwriting dispersion.py


## 4. Cruce Espacial GIS (zonas_gis.py)

In [30]:
%%writefile zonas_gis.py
import requests
from shapely.geometry import Point

def descargar_zonas_cercanas(lat: float, lon: float) -> list:
    """Descarga una sola vez hospitales, escuelas y pueblos en 15 km a la redonda."""
    servidores = [
        "https:" + "//overpass.kumi.systems/api/interpreter",
        "https:" + "//lz4.overpass-api.de/api/interpreter",
        "https:" + "//overpass-api.de/api/interpreter"
    ]
    query = f"""
    [out:json][timeout:15];
    (
      node["amenity"~"hospital|clinic|school|university"](around:15000,{lat},{lon});
      way["amenity"~"hospital|clinic|school|university"](around:15000,{lat},{lon});
      node["place"~"town|village|city"](around:15000,{lat},{lon});
    );
    out center;
    """
    headers = {"User-Agent": "BlastSafePrototype/1.0"}
    zonas = []

    for url in servidores:
        try:
            resp = requests.post(url, data={"data": query}, headers=headers, timeout=12)
            if resp.status_code == 200:
                for el in resp.json().get("elements", []):
                    tags = el.get("tags", {})
                    p_lat = el.get("lat") or el.get("center", {}).get("lat")
                    p_lon = el.get("lon") or el.get("center", {}).get("lon")
                    if p_lat and p_lon:
                        zonas.append({
                            "nombre": tags.get("name", "Zona sin nombre"),
                            "tipo": tags.get("amenity") or tags.get("place") or "zona",
                            "lat": p_lat,
                            "lon": p_lon,
                            "punto": Point(p_lon, p_lat)
                        })
                print(f"Cargadas {len(zonas)} zonas sensibles desde OpenStreetMap.")
                return zonas
        except Exception:
            continue

    print("Aviso: Servidores OSM ocupados, continuando sin capa GIS externa.")
    return zonas

def filtrar_afectados(zonas_cercanas: list, poligono_pluma) -> list:
    if poligono_pluma is None:
        return []
    return [z for z in zonas_cercanas if poligono_pluma.contains(z["punto"])]


Overwriting zonas_gis.py


## 5. Simulador (main.py)


In [31]:
import importlib
import folium
from shapely.geometry import mapping
from IPython.display import HTML
import clima
import dispersion
import zonas_gis

# Recargar módulos por si editaste alguno
importlib.reload(clima)
importlib.reload(dispersion)
importlib.reload(zonas_gis)

# 1. Coordenadas de la Planta de Municiones de Iowa (IAAAP)
LAT, LON = 40.8065, -91.2468

# 2. Obtener pronóstico en vivo (NOAA) y zonas vulnerables (OpenStreetMap)
datos = clima.obtener_pronostico(LAT, LON, horas=24)
zonas_cercanas = zonas_gis.descargar_zonas_cercanas(LAT, LON)

# 3. Extraer variables de la hora actual (índice 0)
hora_actual = next(iter(datos["time"]))
ws10 = next(iter(datos["wind_speed_10m"]))
ws120 = next(iter(datos["wind_speed_120m"]))
wd = next(iter(datos["wind_direction_120m"]))
t2 = next(iter(datos["temperature_2m"]))
t120 = next(iter(datos["temperature_120m"]))

ws = (ws10 + ws120) / 2
inversion = t120 > t2

# 4. Calcular polígono de pluma de plomo y cruzar con zonas sensibles
poligono = dispersion.calcular_pluma(LAT, LON, masa_kg=4400, ws=ws, wd=wd)
zonas_afectadas = zonas_gis.filtrar_afectados(zonas_cercanas, poligono)

print(f"Hora: {hora_actual} | Viento: {ws:.1f} m/s (Origen: {wd}°) | Inversión térmica: {inversion}")
print(f"Zonas vulnerables dentro de la nube: {len(zonas_afectadas)}")
for z in zonas_afectadas:
    print(f"  -> ALERTA: {z['nombre']} ({z['tipo']})")

# 5. Renderizar mapa satelital limpio (Esri World Imagery)
mapa = folium.Map(location=[LAT, LON], zoom_start=12, tiles="Esri.WorldImagery")

# Marcador del punto de detonación
folium.Marker(
    [LAT, LON],
    popup="Punto de Detonación (IAAAP)",
    icon=folium.Icon(color="red", icon="info-sign")
).add_to(mapa)

# Dibujar el cono de dispersión de plomo
if poligono:
    folium.GeoJson(
        mapping(poligono),
        style_function=lambda x: {
            "fillColor": "orange",
            "color": "red",
            "weight": 2,
            "fillOpacity": 0.4
        }
    ).add_to(mapa)

# Dibujar marcadores morados para escuelas, hospitales o pueblos afectados
for z in zonas_afectadas:
    folium.Marker(
        [z["lat"], z["lon"]],
        popup=f"{z['nombre']} ({z['tipo']})",
        icon=folium.Icon(color="purple", icon="exclamation-sign")
    ).add_to(mapa)

HTML(mapa._repr_html_())


Conectado en vivo a NOAA (National Weather Service).
Aviso: Servidores OSM ocupados, continuando sin capa GIS externa.
Hora: 2026-11-25T05:00:00-06:00 | Viento: 2.5 m/s (Origen: 45°) | Inversión térmica: False
Zonas vulnerables dentro de la nube: 0


## 5. Simulador Interactivo 24 Horas


In [27]:
import importlib
import folium
import ipywidgets as widgets
from itertools import islice
from shapely.geometry import mapping
from IPython.display import display, HTML, clear_output
import clima
import dispersion
import zonas_gis

# Recargar módulos por si editaste alguno
importlib.reload(clima)
importlib.reload(dispersion)
importlib.reload(zonas_gis)

# 1. Coordenadas de la Planta de Municiones de Iowa (IAAAP)
LAT, LON = 40.8065, -91.2468

# 2. Descargar pronóstico (24h) y zonas GIS una sola vez al iniciar
datos = clima.obtener_pronostico(LAT, LON, horas=24)
zonas_cercanas = zonas_gis.descargar_zonas_cercanas(LAT, LON)
total_horas = len(datos["time"])

salida_mapa = widgets.Output()

def actualizar_simulacion(indice_hora):
    with salida_mapa:
        clear_output(wait=True)

        # Extraer variables de la hora seleccionada
        hora_str = next(islice(datos["time"], indice_hora, None))
        ws10 = next(islice(datos["wind_speed_10m"], indice_hora, None))
        ws120 = next(islice(datos["wind_speed_120m"], indice_hora, None))
        wd = next(islice(datos["wind_direction_120m"], indice_hora, None))
        t2 = next(islice(datos["temperature_2m"], indice_hora, None))
        t120 = next(islice(datos["temperature_120m"], indice_hora, None))

        ws = (ws10 + ws120) / 2
        inversion = t120 > t2

        # Calcular nube y cruzar con escuelas/hospitales/pueblos
        poligono = dispersion.calcular_pluma(LAT, LON, masa_kg=4400, ws=ws, wd=wd)
        zonas_afectadas = zonas_gis.filtrar_afectados(zonas_cercanas, poligono)

        # Evaluar semáforo de seguridad
        if inversion or len(zonas_afectadas) > 0 or ws < 2.0 or ws > 10.0:
            estado = "NO_GO (ALTO RIESGO)"
            color_cono = "red"
        else:
            estado = "GO (VENTANA SEGURA)"
            color_cono = "green"

        print(f"=== HORA: {hora_str} | ESTADO: {estado} ===")
        print(f"Viento: {ws:.1f} m/s (Origen: {wd}°) | Inversión térmica: {inversion}")
        print(f"Zonas vulnerables dentro de la nube: {len(zonas_afectadas)}")
        for z in zonas_afectadas:
            print(f"  -> ALERTA: {z['nombre']} ({z['tipo']})")

        # Renderizar mapa satelital limpio
        mapa = folium.Map(location=[LAT, LON], zoom_start=11, tiles="Esri.WorldImagery")

        folium.Marker(
            [LAT, LON],
            popup="Punto de Detonación (IAAAP)",
            icon=folium.Icon(color="red", icon="info-sign")
        ).add_to(mapa)

        if poligono:
            folium.GeoJson(
                mapping(poligono),
                style_function=lambda x, c=color_cono: {
                    "fillColor": c,
                    "color": c,
                    "weight": 2,
                    "fillOpacity": 0.45
                }
            ).add_to(mapa)

        for z in zonas_afectadas:
            folium.Marker(
                [z["lat"], z["lon"]],
                popup=f"{z['nombre']} ({z['tipo']})",
                icon=folium.Icon(color="purple", icon="exclamation-sign")
            ).add_to(mapa)

        display(HTML(mapa._repr_html_()))

# Crear control deslizante de 0 a 23 horas
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=max(0, total_horas - 1),
    step=1,
    description="Hora (+h):",
    continuous_update=False,
    layout=widgets.Layout(width="65%")
)

widgets.interact(actualizar_simulacion, indice_hora=slider)
display(salida_mapa)


Conectado en vivo a NOAA (National Weather Service).
Aviso: Servidores OSM ocupados, continuando sin capa GIS externa.


interactive(children=(IntSlider(value=0, continuous_update=False, description='Hora (+h):', layout=Layout(widt…

Output()